# Notebook 40 — L1 Pendulum: Neural State Variables
**Runtime:** Colab T4 GPU. **Prerequisite:** notebook 39.

Replicate nb18c -> nb19 -> nb20: predict t+10 through the BSISO encoder/decoder,
compute the original scikit-dimension ID calls, refine through the estimated d,
and fit a latent dynamics MLP. Also fit d=2 as an explicit ablation.
No physical labels enter these losses. Do not force an ID estimate to equal
2 or 3. A single position-only frame does not identify velocity, so physical
state dimension is not the expected clean snapshot dimension. Compare with
the project NSV pipeline and with same-dimensional SSL outputs.
Full autonomous latent rollouts are not implemented here; notebook 42 evaluates
common direct forecasts and state readouts.


## Protocol shared by notebooks 40 and 41

**climate_bsiso_v1:** replicate each climate method's architecture/training recipe,
using the BSISO 2-D CNN for single-channel 64x64 pendulum images. Methods share
the data split, one-frame observation and evaluation rows, but have different
backbones/heads/optimizers just as the original pipelines do.

InfoNCE: direct 2-D CNN output, raw single-direction loss, temperature=0.5,
positive offsets +/-1..3 frames. BT: NSV-warm-started encoder, nb26 projector,
sum of weighted losses at lags 1..5 (weight 1 -> 0.05). NSV: predict t+10,
estimate ID, fit the nb20/23 raw-latent SIREN, then the latent dynamics MLP.
One frame is T0/50 by default, not one calendar day. NSV lag10 is 0.2T0.
The three climate methods add **no training-view noise**. Optional BT-same is
a toy-only control with independent std0.01 pixel augmentation.

Model selection follows the reference: InfoNCE final epoch; BT minimum training
loss; NSV prediction/refinement minimum validation MSE. Original ID uses train+val
features; independent test trajectories never enter ID/model selection.
No simulator state labels enter representation training. Climate preprocessing
(bandpass/anomalies) and the MJO 1-D spatial encoder are not replicated by images.
A clean fixed-L snapshot depends on theta, not velocity. Velocity readout and
future prediction measure information missing from this observation; do not
require recovery of the full two-state oracle from one position-only frame.
For L1c, visible-length snapshots depend on (theta,L), hidden-length ones on theta.
Earlier common-backbone runs remain historical controls; notebook42 defaults
to this new protocol only. Existing simulator data can be reused; retrain 40/41.
A completed run is reused by exact config hash. Incomplete runs are not overwritten;
inspect them or change `run_tag`. Full rerun/resume of partial epochs is not implemented.


## Cell 1 — Drive and isolated paths


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, sys, tempfile, json

PROJECT_DIR = Path('/content/drive/MyDrive/BSISO_SSL_Project')
L1_ROOT = PROJECT_DIR / 'toy' / 'pendulum'
L1_ROOT.mkdir(parents=True, exist_ok=True)
# Embedded code needs no GitHub clone/download or external .py upload.
RUNTIME_DIR = Path(tempfile.mkdtemp(prefix='ddcs_l1_code_'))
os.chdir(RUNTIME_DIR)
sys.path.insert(0, str(RUNTIME_DIR))
for name in ('l1_data', 'l1_learning', 'l1_climate', 'l1_evaluation'):
    sys.modules.pop(name, None)
CACHE_DIR = Path('/content/ddcs_l1_frames')  # disposable; never store frame arrays on Drive

def resolve_dataset_id(value):
    if value:
        if not (L1_ROOT / 'datasets' / value / 'manifest.json').exists():
            raise FileNotFoundError(f'Dataset {value} not found; run notebook 39 first.')
        return value
    available = sorted(p.parent.name for p in (L1_ROOT / 'datasets').glob('*/manifest.json'))
    if len(available) != 1:
        raise ValueError(f'Set DATASET_ID explicitly from notebook 39. Available: {available}')
    return available[0]

print('Drive outputs:', L1_ROOT)
print('Local code:', RUNTIME_DIR)


## Cell 2 — Embedded implementations and reference ID dependency


In [ ]:
%pip install -q scikit-dimension==0.3.4


In [ ]:
%%writefile l1_data.py
"""L1 pendulum simulation, observation, split and provenance utilities (no torch)."""
from __future__ import annotations

from dataclasses import asdict, dataclass
import hashlib
import json
from pathlib import Path

import numpy as np

SCHEMA = "pendulum-l1-v1"
SPLITS = {"train": 0, "val": 1, "test": 2}


def json_write(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False))
    tmp.replace(path)


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()[:16]


def file_sha(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


@dataclass(frozen=True)
class SimulationConfig:
    seed: int = 20261027
    n_fixed: int = 100
    n_length_groups: int = 60
    trajectories_per_length: int = 5
    periods: float = 8.0
    frames_per_period: int = 50
    substeps: int = 8
    g: float = 1.0
    reference_length: float = 1.0
    damping: float = 0.02
    length_min: float = 0.8
    length_max: float = 1.2
    image_size: int = 64

    def validate(self):
        if self.n_fixed < 5 or self.n_length_groups < 5:
            raise ValueError("Need at least five independent groups for train/val/test.")
        if self.trajectories_per_length < 1 or self.periods <= 0:
            raise ValueError("Invalid trajectory count/duration.")
        if self.frames_per_period < 10 or self.substeps < 1:
            raise ValueError("Insufficient sampling/integration resolution.")
        if self.g <= 0 or self.reference_length <= 0 or self.damping < 0:
            raise ValueError("Invalid physical parameters.")
        if not 0 < self.length_min < self.length_max <= 1.5 * self.reference_length:
            raise ValueError("Length interval must be positive and fit the fixed camera.")
        if self.image_size != 64:
            raise ValueError("The initial CNN/renderer protocol uses 64 x 64 images.")

    @property
    def period(self):
        return 2 * np.pi * np.sqrt(self.reference_length / self.g)

    @property
    def dt(self):
        return self.period / self.frames_per_period


def group_split(groups, seed):
    groups = np.asarray(groups)
    unique = np.unique(groups)
    if len(unique) < 5:
        raise ValueError("Too few independent groups.")
    order = np.random.default_rng(seed).permutation(unique)
    n_val = max(1, int(round(0.2 * len(order))))
    n_test = n_val
    mapping = {int(g): 0 for g in order}
    mapping.update({int(g): 1 for g in order[-n_val-n_test:-n_test]})
    mapping.update({int(g): 2 for g in order[-n_test:]})
    return np.array([mapping[int(g)] for g in groups], dtype=np.int8)


def simulate(initial, lengths, times, g=1.0, damping=0.02, substeps=8):
    """Vectorized RK4, constant length per trajectory; no stochastic force in L1a-c."""
    state = np.asarray(initial, dtype=np.float64).copy()
    lengths = np.asarray(lengths, dtype=np.float64)
    times = np.asarray(times)
    if state.shape != (len(lengths), 2) or np.any(lengths <= 0):
        raise ValueError("Expected initial=(trajectories,2), positive lengths.")
    if substeps < 1 or len(times) < 2 or np.any(np.diff(times) <= 0):
        raise ValueError("Invalid time grid or substeps.")
    out = np.empty((len(lengths), len(times), 2), dtype=np.float64)
    out[:, 0] = state

    def rhs(y):
        return np.stack((y[:, 1], -damping*y[:, 1] - g/lengths*np.sin(y[:, 0])), axis=1)

    for i, interval in enumerate(np.diff(times), start=1):
        h = interval / substeps
        for _ in range(substeps):
            k1 = rhs(state)
            k2 = rhs(state + h*k1/2)
            k3 = rhs(state + h*k2/2)
            k4 = rhs(state + h*k3)
            state += h*(k1 + 2*k2 + 2*k3 + k4)/6
        out[:, i] = state
    return out


def energy(states, lengths, g=1.0):
    """Mechanical energy for unit mass (constant L), not a harmonic approximation."""
    length = np.asarray(lengths)[:, None]
    return 0.5*length**2*states[..., 1]**2 + g*length*(1-np.cos(states[..., 0]))


def make_bundle(cfg, variable_length=False):
    cfg.validate()
    rng = np.random.default_rng(cfg.seed + (1000 if variable_length else 0))
    if variable_length:
        group_lengths = rng.uniform(cfg.length_min, cfg.length_max, cfg.n_length_groups)
        groups = np.repeat(np.arange(cfg.n_length_groups), cfg.trajectories_per_length)
        lengths = group_lengths[groups]
    else:
        groups = np.arange(cfg.n_fixed)
        lengths = np.full(cfg.n_fixed, cfg.reference_length)
    initial = rng.uniform([-1.0, -0.5], [1.0, 0.5], size=(len(groups), 2))
    # Avoid a dataset dominated by almost stationary initial conditions.
    weak = np.linalg.norm(initial, axis=1) < 0.15
    while weak.any():
        initial[weak] = rng.uniform([-1.0, -0.5], [1.0, 0.5], size=(int(weak.sum()), 2))
        weak = np.linalg.norm(initial, axis=1) < 0.15
    times = np.arange(int(round(cfg.periods*cfg.frames_per_period)) + 1)*cfg.dt
    return {
        "states": simulate(initial, lengths, times, cfg.g, cfg.damping, cfg.substeps),
        "times": times, "lengths": lengths, "groups": groups.astype(np.int32),
        "splits": group_split(groups, cfg.seed + 17),
    }


def numerical_checks(bundle, cfg):
    original = bundle["states"][:3]
    fine = simulate(original[:, 0], bundle["lengths"][:3], bundle["times"],
                    cfg.g, cfg.damping, cfg.substeps*2)
    err = float(np.max(np.abs(original-fine)))
    e = energy(bundle["states"], bundle["lengths"], cfg.g)
    max_rise = float(np.max(np.diff(e, axis=1)))
    if err > 1e-5 or max_rise > 1e-8:
        raise ValueError(f"Integrator check failed: convergence={err}, energy rise={max_rise}")
    for group in np.unique(bundle["groups"]):
        if len(np.unique(bundle["splits"][bundle["groups"] == group])) != 1:
            raise ValueError("A group crossed splits.")
    return {"rk4_step_halving_max_error": err, "max_energy_increment": max_rise,
            "n_trajectories": len(e), "frames_per_trajectory": e.shape[1]}


def create_dataset(root, cfg):
    """Small truth arrays on Drive; frames are rendered only into disposable Colab cache."""
    cfg.validate()
    identity = {"schema": SCHEMA, "simulation": asdict(cfg), "source_sha": file_sha(__file__)}
    dataset_id = digest(identity)
    out = Path(root)/"datasets"/dataset_id
    if (out/"manifest.json").exists():
        load_dataset(root, dataset_id, "fixed")  # Validate existing hashes, never overwrite.
        load_dataset(root, dataset_id, "variable")
        return dataset_id
    if out.exists():
        raise FileExistsError(f"Incomplete dataset exists: {out}; use a new seed or inspect it.")
    out.mkdir(parents=True)
    checks, hashes = {}, {}
    for name, variable in [("fixed", False), ("variable", True)]:
        bundle = make_bundle(cfg, variable)
        checks[name] = numerical_checks(bundle, cfg)
        np.savez_compressed(out/f"{name}.npz", **bundle)
        hashes[name] = file_sha(out/f"{name}.npz")
    json_write(out/"manifest.json", {**identity, "dataset_id": dataset_id,
                                    "checks": checks, "data_sha256": hashes})
    return dataset_id


def load_dataset(root, dataset_id, kind):
    if kind not in {"fixed", "variable"}:
        raise ValueError(kind)
    out = Path(root)/"datasets"/dataset_id
    manifest = json.loads((out/"manifest.json").read_text())
    if manifest["schema"] != SCHEMA or manifest["dataset_id"] != dataset_id:
        raise ValueError("Dataset identity/schema mismatch.")
    if manifest["source_sha"] != file_sha(__file__):
        raise ValueError("Data code differs from notebook 39; use the same notebook release.")
    if file_sha(out/f"{kind}.npz") != manifest["data_sha256"][kind]:
        raise ValueError("Dataset file checksum mismatch.")
    with np.load(out/f"{kind}.npz", allow_pickle=False) as data:
        bundle = {k: data[k] for k in data.files}
    return bundle, SimulationConfig(**manifest["simulation"]), manifest


CONDITIONS = {
    "L1a": {"kind": "fixed", "noise_std": 0.0, "visible_length": True},
    "L1b_s002": {"kind": "fixed", "noise_std": 0.02, "visible_length": True},
    "L1b_s005": {"kind": "fixed", "noise_std": 0.05, "visible_length": True},
    "L1c_visible": {"kind": "variable", "noise_std": 0.02, "visible_length": True},
    "L1c_hidden": {"kind": "variable", "noise_std": 0.02, "visible_length": False},
}


def render(theta, lengths, size=64, visible_length=True, reference_length=1.0):
    """Fixed camera, Gaussian anti-aliased rod and bob; returns (N,H,W) in [0,1]."""
    theta = np.atleast_1d(theta)
    lengths = np.broadcast_to(lengths, theta.shape)
    radius = 0.42*(size-1)*(lengths/reference_length if visible_length else np.ones_like(theta))
    px, py = (size-1)/2, 0.14*(size-1)
    dx, dy = radius*np.sin(theta), radius*np.cos(theta)
    yy, xx = np.mgrid[:size, :size]
    rx, ry = xx[None]-px, yy[None]-py
    a = np.clip((rx*dx[:, None, None]+ry*dy[:, None, None]) /
                radius[:, None, None]**2, 0, 1)
    d2 = (rx-a*dx[:, None, None])**2+(ry-a*dy[:, None, None])**2
    rod = 0.45*np.exp(-d2/(2*0.65**2))
    bob = np.exp(-((rx-dx[:, None, None])**2+(ry-dy[:, None, None])**2)/(2*1.65**2))
    pivot = 0.7*np.exp(-(rx**2+ry**2)/(2*1.1**2))
    return np.maximum(np.maximum(rod, bob), pivot).astype(np.float32)


def frame_cache(bundle, cfg, condition, cache_root, dataset_id):
    spec = CONDITIONS[condition]
    name = f"{dataset_id}_{spec['kind']}_{'visible' if spec['visible_length'] else 'hidden'}"
    path = Path(cache_root)/f"{name}.npy"
    shape = (*bundle["states"].shape[:2], cfg.image_size, cfg.image_size)
    if path.exists():
        arr = np.load(path, mmap_mode="r", allow_pickle=False)
        if arr.shape != shape or arr.dtype != np.uint8:
            raise ValueError("Frame cache shape/dtype mismatch; remove this disposable cache.")
        return arr
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".partial.npy")
    arr = np.lib.format.open_memmap(tmp, mode="w+", dtype="uint8", shape=shape)
    for tr in range(shape[0]):
        arr[tr] = np.rint(255*render(bundle["states"][tr, :, 0], bundle["lengths"][tr],
                                   cfg.image_size, spec["visible_length"], cfg.reference_length)).astype("uint8")
    arr.flush()
    del arr
    tmp.replace(path)
    return np.load(path, mmap_mode="r", allow_pickle=False)


def index_windows(bundle, window=5, future=5, stride=2):
    """Each row is (trajectory, final input frame); all windows remain in one trajectory."""
    if min(window, stride) < 1 or future < 0:
        raise ValueError("Invalid window/stride/future.")
    ends = np.arange(window-1, bundle["states"].shape[1]-future, stride)
    if not len(ends):
        raise ValueError("Trajectories are too short for the requested history/horizon.")
    n = len(bundle["states"])
    return np.column_stack((np.repeat(np.arange(n), len(ends)), np.tile(ends, n))).astype("int32")


def observation(frames, trajectory, end, window, noise_std, seed, draw=0):
    """Noise is keyed by frame: overlapping windows share the same observed pixels."""
    start = int(end)-window+1
    if start < 0 or end >= frames.shape[1]:
        raise IndexError("Window outside trajectory.")
    out = np.array(frames[int(trajectory), start:int(end)+1], dtype="float32")/255
    if noise_std:
        for j, t in enumerate(range(start, int(end)+1)):
            rng = np.random.default_rng(np.random.SeedSequence([seed, draw, int(trajectory), t]))
            out[j] += noise_std*rng.standard_normal(out[j].shape).astype("float32")
    # Do not clip or normalize individual frames: keep the stated zero-mean noise model.
    return out


def sample_truth(bundle, index, horizon=0):
    tr, t = index.T
    return np.column_stack((bundle["states"][tr, t+horizon], bundle["lengths"][tr]))


def estimate_id(features, groups, ks=(10, 20, 40), max_points=2000, seed=17):
    """Train-only kNN MLE diagnostic; exclude neighbors from the same trajectory/group.

    This is a finite-sample, scale-dependent estimate, not a guaranteed physical dimension.
    No per-coordinate whitening: it would amplify near-null latent directions.
    """
    from sklearn.neighbors import NearestNeighbors
    z = np.asarray(features, dtype="float64")
    groups = np.asarray(groups)
    if len(z) != len(groups) or z.ndim != 2 or not np.isfinite(z).all():
        raise ValueError("Invalid ID input.")
    take = np.random.default_rng(seed).choice(len(z), min(len(z), max_points), replace=False)
    z, groups = z[take], groups[take]
    candidates = min(len(z), max(128, max(ks)*8))
    distances, neighbors = NearestNeighbors(n_neighbors=candidates).fit(z).kneighbors(z)
    rows = []
    for k in ks:
        estimates = []
        for i in range(len(z)):
            valid = (groups[neighbors[i]] != groups[i]) & (distances[i] > 1e-10)
            d = distances[i][valid][:k]
            if len(d) == k:
                denominator = np.log(d[-1]/d[:-1]).sum()
                if denominator > 1e-10:
                    estimates.append((k-2)/denominator)
        rows.append({"k": int(k), "n_valid": len(estimates),
                     "median_id": float(np.median(estimates)) if estimates else None,
                     "mean_id": float(np.mean(estimates)) if estimates else None})
    singular = np.linalg.svd(z-z.mean(0), compute_uv=False)**2
    pr = float(singular.sum()**2/(np.square(singular).sum()+1e-30))
    return {"n_sampled": len(z), "scales": rows, "participation_ratio": pr,
            "neighbor_exclusion": "same group", "used_split": "train"}


In [ ]:
%%writefile l1_climate.py
"""Climate-pipeline replica for single-frame pendulum images.

BSISO supplies the 2-D encoders/decoder, nb08/15 the raw InfoNCE objective,
nb26 the Barlow projector/lag objective, and nb20/23 the SIREN refinement.
See PIPELINE_PARITY.md for deliberate adaptations; legacy l1_learning is separate.
"""
from __future__ import annotations

from dataclasses import asdict, dataclass, replace
import copy
import json
import math
from pathlib import Path
import platform
import random

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset, TensorDataset

from l1_data import (CONDITIONS, digest, estimate_id, file_sha, frame_cache,
                     index_windows, json_write, load_dataset, observation)

PROTOCOL = "climate_bsiso_v1"


@dataclass(frozen=True)
class ClimateConfig:
    protocol: str = PROTOCOL
    method: str = "infonce"
    seed: int = 42
    window: int = 1
    train_stride: int = 1
    eval_stride: int = 4
    hidden_dim: int = 64
    projection_dim: int = 2
    lag: int = 10                     # NSV prediction/dynamics only
    positive_radius: int = 3          # InfoNCE: +/- radius, excluding zero
    barlow_lags: tuple = (1, 2, 3, 4, 5)
    barlow_offdiag: float = 0.005
    barlow_min_weight: float = 0.05
    temperature: float = 0.5
    view_noise_std: float = 0.0
    batch_size: int = 64
    epochs: int = 50
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    min_learning_rate: float = 1e-5
    gradient_clip: float = 1.0
    checkpoint_selection: str = "final_epoch"
    warm_start: bool = False
    val_pairs: int = 1000
    refine_epochs: int = 500
    refine_batch_size: int = 64
    refine_learning_rate: float = 1e-4
    refine_min_learning_rate: float = 1e-6
    refine_dims: tuple = (2,)         # extra ablations, in addition to estimated d
    id_backend: str = "skdim"
    id_scope: str = "train_val"      # nb19/22; independent test remains excluded
    id_max_samples: int = 5000
    dynamics_epochs: int = 300
    run_tag: str = "climate_replica"

    def validate(self):
        if self.protocol != PROTOCOL or self.window != 1 or self.hidden_dim != 64:
            raise ValueError("Climate replica requires its explicit protocol, window=1, hidden_dim=64.")
        if self.method not in {"infonce", "nsv", "bt_temporal", "bt_same"}:
            raise ValueError(self.method)
        if min(self.train_stride, self.eval_stride, self.projection_dim, self.lag,
               self.positive_radius, self.epochs, self.refine_epochs, self.refine_batch_size,
               self.val_pairs, self.id_max_samples) < 1 or self.batch_size < 2:
            raise ValueError("Invalid sample counts or training dimensions.")
        if not self.barlow_lags or min(self.barlow_lags) < 1 or len(set(self.barlow_lags)) != len(self.barlow_lags):
            raise ValueError("Barlow lags must be distinct positive integers.")
        if self.temperature <= 0 or not 0 < self.barlow_min_weight <= 1:
            raise ValueError("Invalid temperature/lag weighting.")
        if min(self.learning_rate, self.refine_learning_rate) <= 0 or min(self.view_noise_std, self.weight_decay, self.barlow_offdiag, self.gradient_clip, self.dynamics_epochs) < 0:
            raise ValueError("Invalid optimizer or augmentation settings.")
        if any(d < 1 for d in self.refine_dims):
            raise ValueError("Refiner dimensions must be positive.")
        if self.id_backend not in {"skdim", "group_mle"} or self.id_scope not in {"train", "train_val"}:
            raise ValueError("Unknown ID protocol.")
        if self.checkpoint_selection not in {"final_epoch", "validation_objective", "training_objective"}:
            raise ValueError("Unknown checkpoint selection.")
        if self.warm_start and not self.method.startswith("bt_"):
            raise ValueError("Only Barlow uses the NSV warm start.")


def climate_config(method, **overrides):
    """Method-specific notebook defaults, not a common optimizer/architecture."""
    cfg = ClimateConfig(method=method)
    if method == "nsv":
        cfg = replace(cfg, epochs=100, gradient_clip=0.0, checkpoint_selection="validation_objective")
    elif method.startswith("bt_"):
        cfg = replace(cfg, projection_dim=3, batch_size=256, epochs=100, gradient_clip=0.0,
                      checkpoint_selection="training_objective", warm_start=True,
                      view_noise_std=0.01 if method == "bt_same" else 0.0)
    cfg = replace(cfg, **overrides)
    cfg.validate()
    return cfg


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


class SSL2DEncoder(nn.Module):
    """nb08: only input channels change from three climate fields to one image."""
    def __init__(self, embedding_dim=2):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(16); self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(32); self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(32, 32, 3, padding=1, bias=False)
        self.bn3 = nn.BatchNorm2d(32)
        self.global_pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Linear(32, embedding_dim)
        initialize_bsiso(self)

    def pre_fc(self, x):
        x = self.pool1(F.relu(self.bn1(self.conv1(x))))
        x = self.pool2(F.relu(self.bn2(self.conv2(x))))
        return self.global_pool(F.relu(self.bn3(self.conv3(x)))).flatten(1)

    def forward(self, x):
        return self.fc(self.pre_fc(x))


def initialize_bsiso(model, decoder=False):
    for layer in model.modules():
        if isinstance(layer, nn.Conv2d):
            nn.init.kaiming_normal_(layer.weight, mode="fan_out", nonlinearity="relu")
            if decoder and layer.bias is not None:
                nn.init.constant_(layer.bias, 0)
        elif isinstance(layer, nn.BatchNorm2d):
            nn.init.constant_(layer.weight, 1); nn.init.constant_(layer.bias, 0)
        elif isinstance(layer, nn.Linear):
            nn.init.normal_(layer.weight, 0, 0.01); nn.init.constant_(layer.bias, 0)


class PredictiveEncoder(nn.Module):
    """nb18c BSISO encoder; input channel adaptation only."""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 32, 4, 2, 1, bias=False); self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, 3, 1, 1, bias=False); self.bn2 = nn.BatchNorm2d(32)
        self.conv3 = nn.Conv2d(32, 64, 4, 2, 1, bias=False); self.bn3 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, 3, 1, 1, bias=False); self.bn4 = nn.BatchNorm2d(64)
        self.conv5 = nn.Conv2d(64, 128, 4, 2, 1, bias=False); self.bn5 = nn.BatchNorm2d(128)
        self.gap = nn.AdaptiveAvgPool2d(1); self.fc = nn.Linear(128, latent_dim)
        initialize_bsiso(self)

    def forward(self, x):
        for i in range(1, 6):
            x = F.relu(getattr(self, f"bn{i}")(getattr(self, f"conv{i}")(x)))
        return self.fc(self.gap(x).flatten(1))


class PredictiveDecoder(nn.Module):
    """nb18c decoder; resize ladder adapted to 64x64, final channel to one."""
    def __init__(self, latent_dim=64):
        super().__init__()
        self.fc = nn.Linear(latent_dim, 128)
        self.conv1 = nn.Conv2d(128, 64, 3, padding=1, bias=False); self.bn1 = nn.BatchNorm2d(64)
        self.conv2 = nn.Conv2d(64, 64, 3, padding=1, bias=False); self.bn2 = nn.BatchNorm2d(64)
        self.conv3 = nn.Conv2d(64, 32, 3, padding=1, bias=False); self.bn3 = nn.BatchNorm2d(32)
        self.conv4 = nn.Conv2d(32, 1, 3, padding=1)
        initialize_bsiso(self, decoder=True)

    def forward(self, h):
        x = self.fc(h).view(-1, 128, 1, 1)
        for i, size in enumerate((8, 16, 32), 1):
            x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
            x = F.relu(getattr(self, f"bn{i}")(getattr(self, f"conv{i}")(x)))
        return self.conv4(F.interpolate(x, size=(64, 64), mode="bilinear", align_corners=False))


class Projector(nn.Module):
    """Exact nb26/27 projector: 64 -> 128 -> 64 -> D, BN/ReLU."""
    def __init__(self, in_dim=64, out_dim=3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Linear(128, 64), nn.BatchNorm1d(64), nn.ReLU(inplace=True),
            nn.Linear(64, out_dim))

    def forward(self, x):
        return self.net(x)


class ClimateModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.method = cfg.method
        if cfg.method == "infonce":
            self.encoder = SSL2DEncoder(cfg.projection_dim)
        else:
            self.encoder = PredictiveEncoder(cfg.hidden_dim)
            if cfg.method == "nsv":
                self.decoder = PredictiveDecoder(cfg.hidden_dim)
            else:
                self.projector = Projector(cfg.hidden_dim, cfg.projection_dim)

    def forward(self, x):
        if self.method == "infonce":
            pooled = self.encoder.pre_fc(x)
            return pooled, self.encoder.fc(pooled)
        h = self.encoder(x)
        return h, self.decoder(h) if self.method == "nsv" else self.projector(h)


def infonce_loss_raw(z_a, z_b, temperature=0.5):
    """nb08/15: one direction, raw dot product, no nearby-negative mask."""
    logits = torch.matmul(z_a, z_b.T) / temperature
    return F.cross_entropy(logits, torch.arange(len(z_a), device=z_a.device))


def barlow_cross_corr(z_a, z_b, eps=1e-5):
    """Preserve nb26's sample std (B-1) and eps outside the square root."""
    if len(z_a) < 2:
        raise ValueError("Barlow correlation needs at least two paired samples.")
    a = (z_a-z_a.mean(0))/(z_a.std(0)+eps)
    b = (z_b-z_b.mean(0))/(z_b.std(0)+eps)
    return a.T @ b / len(a)


def barlow_loss(c, weight=1.0, offdiag=0.005):
    on = (torch.diagonal(c)-1).square().sum()
    off = (c-torch.diag(torch.diagonal(c))).square().sum()
    return weight*(on+offdiag*off)


def lag_weight(lag, cfg):
    lo, hi = min(cfg.barlow_lags), max(cfg.barlow_lags)
    return cfg.barlow_min_weight**((lag-lo)/(hi-lo)) if hi != lo else 1.0


class PairDataset(Dataset):
    """Time-only pairs; never inspect simulated theta, velocity, or length values."""
    def __init__(self, frames, index, condition, data_seed, cfg, training, lag=None):
        self.frames, self.index = frames, np.asarray(index)
        self.spec, self.seed, self.cfg = CONDITIONS[condition], data_seed, cfg
        self.training, self.lag, self.epoch = training, lag, 0
        self.allowed = {(int(tr), int(t)) for tr, t in self.index}
        if cfg.method == "infonce":
            self.neighbors = []
            for tr, t in self.index:
                candidates = [int(t+d) for d in range(-cfg.positive_radius, cfg.positive_radius+1)
                              if d and (int(tr), int(t+d)) in self.allowed]
                if not candidates:
                    raise ValueError("InfoNCE anchor has no temporal neighbor; reduce train_stride.")
                self.neighbors.append(candidates)
            if not training:
                rng = np.random.default_rng(cfg.seed)
                # Same validation sample limit as nb08/15, but reproducible pair RNG.
                chosen = rng.choice(len(self.index), min(cfg.val_pairs, len(self.index)), replace=False)
                self.fixed_pairs = [(int(i), int(rng.choice(self.neighbors[i]))) for i in chosen]

    def __len__(self):
        if self.cfg.method == "infonce" and not self.training:
            return len(self.fixed_pairs)
        return len(self.index)

    def pair_at(self, i):
        if self.cfg.method == "infonce":
            if not self.training:
                i, target = self.fixed_pairs[i]
            else:
                rng = np.random.default_rng(np.random.SeedSequence([self.cfg.seed, self.epoch, int(i), 751]))
                target = int(rng.choice(self.neighbors[i]))
        else:
            target = int(self.index[i, 1]) + int(self.lag)
        tr, t = map(int, self.index[i])
        return tr, t, target

    def __getitem__(self, i):
        tr, t, target = self.pair_at(i)
        x = observation(self.frames, tr, t, 1, self.spec["noise_std"], self.seed, 0)
        # Climate fields are fixed observations reused in either branch.
        draw = 1 if self.cfg.method == "bt_same" else 0
        y = observation(self.frames, tr, target, 1, self.spec["noise_std"], self.seed, draw)
        if self.cfg.view_noise_std:
            rng = np.random.default_rng(np.random.SeedSequence([self.cfg.seed, self.epoch, int(i), 752]))
            x += self.cfg.view_noise_std*rng.standard_normal(x.shape).astype("float32")
            if self.cfg.method != "nsv":
                y += self.cfg.view_noise_std*rng.standard_normal(y.shape).astype("float32")
        return torch.from_numpy(x), torch.from_numpy(y)


def make_loaders(frames, bundle, condition, sim, cfg):
    lags = cfg.barlow_lags if cfg.method == "bt_temporal" else (0 if cfg.method in {"infonce", "bt_same"} else cfg.lag,)
    loaders, counts = {}, {}
    for split in (0, 1):
        loaders[split] = {}
        for lag in lags:
            ix = index_windows(bundle, 1, lag, cfg.train_stride)
            ix = ix[bundle["splits"][ix[:, 0]] == split]
            ds = PairDataset(frames, ix, condition, sim.seed, cfg, split == 0, lag)
            if len(ds) < 2:
                raise ValueError("Need at least two pairs in each split/lag.")
            batch = min(cfg.batch_size, len(ds))
            # BT's projector BatchNorm1d cannot train on a singleton tail.
            drop = split == 0 and cfg.method.startswith("bt_")
            loaders[split][lag] = DataLoader(ds, batch_size=batch, shuffle=split == 0,
                                            drop_last=drop, num_workers=0)
            counts[f"{split}:{lag}"] = {"pairs": len(ds), "batches": len(loaders[split][lag]), "drop_last": drop}
    return loaders, counts


def loss_on_pair(model, pair, cfg, lag, device):
    x, y = (v.to(device) for v in pair)
    _, a = model(x)
    if cfg.method == "nsv":
        return F.mse_loss(a, y), None
    _, b = model(y)
    if cfg.method == "infonce":
        norms = torch.cat((a.detach().norm(dim=1), b.detach().norm(dim=1)))
        return infonce_loss_raw(a, b, cfg.temperature), norms
    c = barlow_cross_corr(a, b)
    weight = lag_weight(lag, cfg) if cfg.method == "bt_temporal" else 1.0
    return barlow_loss(c, weight, cfg.barlow_offdiag), c


def train_stage1(model, loaders, cfg, device, out):
    optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate, weight_decay=cfg.weight_decay)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, cfg.epochs, eta_min=cfg.min_learning_rate)
    history, best, chosen, selected_epoch = [], math.inf, None, 0
    correlations = {lag: [] for lag in loaders[1]}
    fixed = {lag: next(iter(loader)) for lag, loader in loaders[1].items()}

    def record_correlations():
        if not cfg.method.startswith("bt_"):
            return
        model.eval()
        with torch.no_grad():
            for lag, pair in fixed.items():
                _, c = loss_on_pair(model, pair, cfg, lag, device)
                correlations[lag].append(c.cpu().numpy())

    record_correlations()
    for epoch in range(cfg.epochs):
        for loader in loaders[0].values():
            loader.dataset.epoch = epoch
        record = {"epoch": epoch+1, "learning_rate": optimizer.param_groups[0]["lr"]}
        model.train()
        iters = {lag: iter(loader) for lag, loader in loaders[0].items()}
        steps = min(len(loader) for loader in loaders[0].values())
        total_value, total_weight = 0.0, 0
        norm_parts, on_by_lag, off_by_lag = [], {}, {}
        for _ in range(steps):
            optimizer.zero_grad()
            terms, batch_n = [], 0
            for lag, iterator in iters.items():
                pair = next(iterator); batch_n = len(pair[0])
                loss, diagnostic = loss_on_pair(model, pair, cfg, lag, device)
                if cfg.method == "infonce":
                    norm_parts.append(diagnostic.cpu().numpy())
                elif cfg.method.startswith("bt_"):
                    c = diagnostic.detach()
                    on_by_lag[lag] = on_by_lag.get(lag, 0.0)+float((c.diag()-1).square().mean())
                    off = (c-torch.diag(c.diag())).square().sum()
                    off_by_lag[lag] = off_by_lag.get(lag, 0.0)+float(off)/max(1, len(c)*(len(c)-1))
                terms.append(loss)
            loss = torch.stack(terms).sum()  # nb26: all lag losses, ONE backward/step
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite {cfg.method} loss at epoch {epoch+1}.")
            loss.backward()
            if cfg.gradient_clip:
                torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.gradient_clip)
            optimizer.step()
            # nb08 averages batch losses; nb18c weights by sample count; BT batches are full.
            weight = batch_n if cfg.method == "nsv" else 1
            total_value += float(loss.detach())*weight; total_weight += weight
        record["train"] = total_value/total_weight
        if norm_parts:
            norms = np.concatenate(norm_parts)
            record.update(mean_norm=float(norms.mean()), std_norm=float(norms.std()), max_norm=float(norms.max()))
        for lag in on_by_lag:
            record[f"on_tau{lag}"] = on_by_lag[lag]/steps
            record[f"off_tau{lag}"] = off_by_lag[lag]/steps
        model.eval()
        with torch.no_grad():
            val = 0.0
            for lag, loader in loaders[1].items():
                values, weights = [], []
                for pair in loader:
                    if cfg.method.startswith("bt_") and len(pair[0]) < 2:
                        continue  # sample std is undefined for a singleton validation batch
                    loss, _ = loss_on_pair(model, pair, cfg, lag, device)
                    values.append(float(loss))
                    weights.append(len(pair[0]) if cfg.method == "nsv" else 1)
                val += float(np.average(values, weights=weights))
            record["val"] = val
        scheduler.step()
        if not math.isfinite(record["val"]):
            raise FloatingPointError("Nonfinite validation loss.")
        key = record["val"] if cfg.checkpoint_selection == "validation_objective" else record["train"]
        if cfg.checkpoint_selection == "final_epoch" or key < best:
            best, chosen, selected_epoch = key, copy.deepcopy(model.state_dict()), epoch+1
        history.append(record)
        record_correlations()
        json_write(out/"training_history.json", history)
        if epoch == 0 or (epoch+1) % 10 == 0 or epoch+1 == cfg.epochs:
            print(f"{cfg.method} epoch {epoch+1}: train={record['train']:.5f} val={record['val']:.5f}", flush=True)
    torch.save(model.state_dict(), out/"model_final.pt")
    model.load_state_dict(chosen); model.eval()
    torch.save(model.state_dict(), out/"model_selected.pt")
    torch.save(model.encoder.state_dict(), out/"encoder_selected.pt")
    json_write(out/"checkpoint_selection.json", {"policy": cfg.checkpoint_selection, "epoch": selected_epoch})
    if cfg.method.startswith("bt_"):
        np.savez_compressed(out/"bt_corr_history.npz", lags=np.array(list(correlations)),
                            **{f"C_tau{lag}": np.array(values) for lag, values in correlations.items()})


class SirenLayer(nn.Module):
    def __init__(self, in_features, out_features, omega_0=30.0, is_first=False):
        super().__init__()
        self.omega_0 = omega_0
        self.linear = nn.Linear(in_features, out_features)
        bound = 1/in_features if is_first else math.sqrt(6/in_features)/omega_0
        with torch.no_grad():
            self.linear.weight.uniform_(-bound, bound)
            self.linear.bias.uniform_(-1e-3, 1e-3)

    def forward(self, x):
        return torch.sin(self.omega_0*self.linear(x))


class SirenAutoencoder(nn.Module):
    """nb20/23 architecture and initialization; raw h, no standardization."""
    def __init__(self, z_dim=64, d_hat=2, omega_0=30.0):
        super().__init__()
        self.e1 = SirenLayer(z_dim, 128, omega_0, True)
        self.e2 = SirenLayer(128, 64, omega_0)
        self.e3 = SirenLayer(64, 32, omega_0)
        self.e4 = nn.Linear(32, d_hat)
        self.d1 = SirenLayer(d_hat, 32, omega_0, True)
        self.d2 = SirenLayer(32, 64, omega_0)
        self.d3 = SirenLayer(64, 128, omega_0)
        self.d4 = nn.Linear(128, z_dim)
        nn.init.kaiming_uniform_(self.e4.weight, nonlinearity="linear"); nn.init.zeros_(self.e4.bias)
        nn.init.kaiming_uniform_(self.d4.weight, nonlinearity="linear"); nn.init.zeros_(self.d4.bias)

    def encode(self, h):
        return self.e4(self.e3(self.e2(self.e1(h))))

    def decode(self, z):
        return self.d4(self.d3(self.d2(self.d1(z))))

    def forward(self, h):
        return self.decode(self.encode(h))


class DynamicsMLP(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 32), nn.ReLU(), nn.Linear(32, 64), nn.ReLU(),
                                 nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, d))

    def forward(self, x):
        return self.net(x)


def fit_array_regression(model, x, y, splits, epochs, batch_size, lr, eta_min, wd, device, out, name):
    """Train/val only; test features never enter the optimizer or checkpoint selection."""
    loaders = {s: DataLoader(TensorDataset(torch.from_numpy(x[splits == s]), torch.from_numpy(y[splits == s])),
                             batch_size=batch_size, shuffle=s == 0) for s in (0, 1)}
    if not all(len(loader.dataset) for loader in loaders.values()):
        raise ValueError("Missing train/validation regression samples.")
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=eta_min)
    history, best, state = [], math.inf, None
    for epoch in range(epochs):
        row = {"epoch": epoch+1}
        for split, label in [(0, "train"), (1, "val")]:
            model.train(split == 0); total, count = 0.0, 0
            for a, b in loaders[split]:
                a, b = a.to(device), b.to(device)
                with torch.set_grad_enabled(split == 0):
                    loss = F.mse_loss(model(a), b)
                    if not torch.isfinite(loss):
                        raise FloatingPointError(f"Nonfinite {name} loss.")
                    if split == 0:
                        opt.zero_grad(); loss.backward(); opt.step()
                total += float(loss.detach())*len(a); count += len(a)
            row[label] = total/count
        scheduler.step(); history.append(row)
        if row["val"] < best:
            best, state = row["val"], copy.deepcopy(model.state_dict())
        if epoch == 0 or (epoch+1) % 25 == 0 or epoch+1 == epochs:
            json_write(out/f"{name}_history.json", history)
            print(f"{name} epoch {epoch+1}: train={row['train']:.6g} val={row['val']:.6g}", flush=True)
    model.load_state_dict(state); model.eval()
    torch.save(state, out/f"{name}.pt")
    json_write(out/f"{name}_history.json", history)
    return model


def encode_frames(model, frames, index, condition, sim, device, batch_size):
    h_parts, z_parts = [], []
    model.eval()
    spec = CONDITIONS[condition]
    with torch.no_grad():
        for chunk in np.array_split(index, max(1, math.ceil(len(index)/batch_size))):
            x = np.stack([observation(frames, tr, t, 1, spec["noise_std"], sim.seed, 0) for tr, t in chunk])
            h, z = model(torch.from_numpy(x).to(device))
            h_parts.append(h.cpu().numpy())
            if model.method != "nsv":
                z_parts.append(z.cpu().numpy())
    return np.concatenate(h_parts), np.concatenate(z_parts) if z_parts else None


def apply_numpy(model, x, device, encode=False, batch_size=512):
    model.eval()
    with torch.no_grad():
        return np.concatenate([(model.encode if encode else model)(torch.from_numpy(chunk).to(device)).cpu().numpy()
                               for chunk in np.array_split(x, max(1, math.ceil(len(x)/batch_size)))])


def climate_id(h, splits, groups, cfg):
    mask = splits == 0 if cfg.id_scope == "train" else splits != 2
    data = h[mask]
    if cfg.id_backend == "group_mle":
        diagnostic = estimate_id(data, groups[mask], seed=cfg.seed)
        values = [row["median_id"] for row in diagnostic["scales"] if row["median_id"] is not None]
        if not values:
            raise ValueError("Insufficient groups/samples for ID.")
        return {"d_hat": max(1, round(float(np.mean(values)))), "backend": "group_mle_ablation",
                "scope": cfg.id_scope, "diagnostic": diagnostic}
    import skdim
    # nb19/22 deduplication and fractional-k sweep, with a recorded sample cap.
    _, unique = np.unique(data, axis=0, return_index=True)
    data = data[np.sort(unique)]
    n_unique = len(data)
    rng = np.random.default_rng(cfg.seed)
    if len(data) > cfg.id_max_samples:
        data = data[np.sort(rng.choice(len(data), cfg.id_max_samples, replace=False))]
    if len(data) <= 20:
        raise ValueError("Too few unique points for the climate ID estimator; use a larger dataset.")
    ks = [max(int(len(data)*fraction), 3) for fraction in (.008, .010, .012, .014, .016)]
    values = [float(skdim.id.MLE(K=k).fit(data).dimension_) for k in ks]
    # Record an explicit-neighbor sweep too: useful for auditing the old K argument.
    explicit = [float(skdim.id.MLE().fit(data, n_neighbors=k).dimension_) for k in ks]
    tnn = float(skdim.id.TwoNN().fit(data).dimension_)
    lpca = float(skdim.id.lPCA().fit(data).dimension_)
    noise = rng.standard_normal(data.shape).astype("float32")
    shuffled = data.copy()
    for column in range(shuffled.shape[1]):
        rng.shuffle(shuffled[:, column])
    controls = {"noise_TwoNN": float(skdim.id.TwoNN().fit(noise).dimension_),
                "noise_MLE": float(skdim.id.MLE(K=ks[2]).fit(noise).dimension_),
                "shuffled_TwoNN": float(skdim.id.TwoNN().fit(shuffled).dimension_),
                "shuffled_MLE": float(skdim.id.MLE(K=ks[2]).fit(shuffled).dimension_)}
    values_all = values+explicit+[tnn, lpca]+list(controls.values())
    if not np.isfinite(values_all).all():
        raise ValueError("Nonfinite ID estimates; inspect the encoder before refinement.")
    return {"backend": "skdim_0.3.4", "scope": cfg.id_scope, "n_unique": n_unique,
            "n_sampled": len(data), "k_list": ks, "LB_by_k_legacy_call": values,
            "LB_by_explicit_n_neighbors": explicit, "LB_mean": float(np.mean(values)),
            "LB_std": float(np.std(values)), "d_hat": max(1, round(float(np.mean(values)))),
            "TwoNN": tnn, "lPCA": lpca, **controls,
            "dimension_selection": "round mean of original MLE(K=k).fit calls; not proof of physical state dimension"}


def resolve_warm_start(root, dataset_id, condition, cfg, requested=None):
    candidates = []
    base = Path(root)/"runs"/dataset_id/condition
    for path in sorted(base.glob("*/complete.json")):
        meta = json.loads((path.parent/"run.json").read_text())
        train = meta["training"]
        if (meta["method"] == "nsv" and train.get("protocol") == cfg.protocol
                and train["seed"] == cfg.seed and train["window"] == cfg.window
                and train["hidden_dim"] == cfg.hidden_dim
                and meta.get("learning_source_sha") == file_sha(__file__)):
            if requested is None or path.parent.name == requested:
                candidates.append(path.parent)
    if len(candidates) != 1:
        raise ValueError(f"Choose exactly one compatible NSV run for warm start; run 40 first. Candidates: {[p.name for p in candidates]}")
    path = candidates[0]
    complete = json.loads((path/"complete.json").read_text())
    sha = file_sha(path/"encoder_selected.pt")
    if sha != complete["encoder_sha256"]:
        raise ValueError("NSV encoder checkpoint checksum mismatch.")
    return path, {"run_id": path.name, "encoder_sha256": sha}


def run_training(root, dataset_id, condition, cache_root, cfg, device=None, warm_start_run_id=None):
    cfg.validate()
    if cfg.method == "nsv" and cfg.id_backend == "skdim":
        try:
            from importlib.metadata import version
            installed = version("scikit-dimension")
        except Exception as exc:
            raise ImportError("Run notebook 40's Colab-only scikit-dimension==0.3.4 install cell first.") from exc
        if installed != "0.3.4":
            raise ValueError(f"Expected scikit-dimension 0.3.4, found {installed}.")
    spec = CONDITIONS[condition]
    bundle, sim, manifest = load_dataset(root, dataset_id, spec["kind"])
    warm_path, warm_identity = None, None
    if cfg.warm_start:
        warm_path, warm_identity = resolve_warm_start(root, dataset_id, condition, cfg, warm_start_run_id)
    references = json.loads(Path(__file__).with_name("climate_reference_manifest.json").read_text())
    identity = {"dataset_id": dataset_id, "condition": condition, "condition_spec": spec,
                "method": cfg.method, "training": asdict(cfg), "warm_start": warm_identity,
                "data_sha": manifest["data_sha256"][spec["kind"]],
                "learning_source_sha": file_sha(__file__), "reference_manifest": references}
    run_id = f"{cfg.method}_s{cfg.seed}_{digest(identity)}"
    out = Path(root)/"runs"/dataset_id/condition/run_id
    if (out/"complete.json").exists():
        complete = json.loads((out/"complete.json").read_text())
        for name, key in [("features.npz", "features_sha256"), ("encoder_selected.pt", "encoder_sha256"),
                          ("model_selected.pt", "model_sha256")]:
            if file_sha(out/name) != complete[key]:
                raise ValueError(f"Completed artifact checksum mismatch: {name}")
        print("Reusing completed run:", out)
        return out
    if out.exists():
        raise FileExistsError(f"Incomplete run at {out}; inspect it or use a new run_tag.")
    seed_all(cfg.seed)
    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))
    frames = frame_cache(bundle, sim, condition, cache_root, dataset_id)
    loaders, counts = make_loaders(frames, bundle, condition, sim, cfg)
    model = ClimateModel(cfg).to(device)
    if warm_path:
        model.encoder.load_state_dict(torch.load(warm_path/"encoder_selected.pt", map_location=device))
    out.mkdir(parents=True)
    torch.save(model.encoder.state_dict(), out/"encoder_initial.pt")
    json_write(out/"run.json", {**identity, "run_id": run_id, "pair_counts": counts,
               "initial_encoder_sha256": file_sha(out/"encoder_initial.pt"),
               "runtime": {"python": platform.python_version(), "torch": str(torch.__version__),
                           "numpy": np.__version__, "device": str(device)},
               "feature_semantics": "InfoNCE pre_fc=32D pooled features, z=direct encoder output; BT h=64D, z=projector; NSV z=ID-selected refiner",
               "temporal_units": "frames (T0/50 by default), not calendar days"})
    train_stage1(model, loaders, cfg, device, out)
    index = index_windows(bundle, 1, 2*sim.frames_per_period, cfg.eval_stride)
    if cfg.method != "nsv":
        h, z = encode_frames(model, frames, index, condition, sim, device, cfg.batch_size)
        features = {"index": index, "pre_fc" if cfg.method == "infonce" else "h": h, "z": z}
        json_write(out/"id_train.json", estimate_id(z[bundle["splits"][index[:, 0]] == 0],
                                                   bundle["groups"][index[:, 0]][bundle["splits"][index[:, 0]] == 0], seed=cfg.seed))
    else:
        # Dense anchors preserve the original daily refiner/dynamics sampling.
        dense = index_windows(bundle, 1, cfg.lag, cfg.train_stride)
        h, _ = encode_frames(model, frames, dense, condition, sim, device, cfg.batch_size)
        splits, groups = bundle["splits"][dense[:, 0]], bundle["groups"][dense[:, 0]]
        diagnostic = estimate_id(h[splits == 0], groups[splits == 0], seed=cfg.seed)
        json_write(out/"id_train.json", diagnostic)
        selected = climate_id(h, splits, groups, cfg)
        json_write(out/"intrinsic_dim.json", selected)
        d_hat = selected["d_hat"]
        lookup = {tuple(row): i for i, row in enumerate(dense)}
        try:
            common = np.array([lookup[tuple(row)] for row in index])
        except KeyError as exc:
            raise ValueError("NSV train_stride/lag leaves gaps in the common evaluation index; use train_stride=1.") from exc
        features = {"index": index, "h": h[common]}
        dense_features = {"index": dense, "h": h}
        for dim in sorted(set((d_hat, *cfg.refine_dims))):
            seed_all(cfg.seed+dim)
            refiner = SirenAutoencoder(d_hat=dim)
            fit_array_regression(refiner, h, h, splits, cfg.refine_epochs, cfg.refine_batch_size,
                                 cfg.refine_learning_rate, cfg.refine_min_learning_rate, 0.0,
                                 device, out, f"refine_d{dim}")
            z = apply_numpy(refiner, h, device, encode=True)
            name = "z" if dim == d_hat else f"z_d{dim}"
            features[name] = z[common]; dense_features[name] = z
            if dim == d_hat and cfg.dynamics_epochs:
                pairs = [(i, lookup[(int(tr), int(t)+cfg.lag)]) for i, (tr, t) in enumerate(dense)
                         if (int(tr), int(t)+cfg.lag) in lookup]
                a, b = np.asarray(pairs).T
                seed_all(cfg.seed)
                dynamics = DynamicsMLP(dim)
                fit_array_regression(dynamics, z[a], z[b], splits[a], cfg.dynamics_epochs, 64,
                                     1e-3, 1e-5, 1e-4, device, out, "dynamics")
                prediction = apply_numpy(dynamics, z[a], device)
                scores = {str(split): {"mse": float(np.mean((prediction[splits[a] == split]-z[b][splits[a] == split])**2)),
                                      "persistence_mse": float(np.mean((z[a][splits[a] == split]-z[b][splits[a] == split])**2))}
                          for split in (1, 2)}
                json_write(out/"dynamics_scores.json", {"lag_frames": cfg.lag, "dimension": dim,
                           "forecast_type": "one-step latent MLP, not autonomous rollout", "splits": scores})
        np.savez_compressed(out/"dense_features.npz", **dense_features)
        json_write(out/"refinement_summary.json", {"selected_dimension": d_hat, "extra_dimensions": list(cfg.refine_dims),
                   "target": "raw unstandardized h64", "selection": "minimum validation h-reconstruction MSE"})
    np.savez_compressed(out/"features.npz", **features)
    json_write(out/"complete.json", {"run_id": run_id, "features_sha256": file_sha(out/"features.npz"),
               "encoder_sha256": file_sha(out/"encoder_selected.pt"), "model_sha256": file_sha(out/"model_selected.pt")})
    return out


In [ ]:
%%writefile climate_reference_manifest.json
{
  "protocol": "climate_bsiso_v1",
  "cell_indexing": "zero-based JSON cell indices",
  "source_cell_sha256": {
    "extension_2d/08_ssl_temporal_2d.ipynb": {
      "2": "204e7e941c792bc1942374088d62920fa167fc0fb21f930811b319ace7aed63a",
      "10": "4afec46023d6e5c891203d0fd612f08f6acb629262cf411695459bc8899a215c",
      "12": "4cd9f7beb1bc1ed5382bb1fcaaac2ca08a92d9380a85caae262570819dc19023",
      "14": "08f606d44d70badff0911028f417f3ba9bef27dd80823dc01a868df60cb8cd57"
    },
    "nsv/18c_nsv_bsiso_stage1_lag10.ipynb": {
      "2": "879dc5b01567dd1708a27d50265407b7c6f69180392d270073e9873c2b6c84cd",
      "6": "b1f533b3e1a8bbce162f6cc42aa3c005e763073932836f0d15e3bd12ba1d1363",
      "8": "f85d52c21284f642539d1db7b04135a6d9d245d76fa4e65bef677bbb908cdabf",
      "10": "268697f0a33cef9b049e2c70b4bf60bc6404f4ba71126d05d31ecc1f3417799f"
    },
    "nsv/19_nsv_bsiso_id_estimation.ipynb": {
      "2": "643a78bcc566fdcc97bc2427541e81258c6143ea3e177e429babea00d4fac714",
      "4": "98805564a68429e3e2dfc3b9ad81ab710ed264320910eb46e1765bf4b22699c3",
      "6": "a65771bc95c9bca55d648da260876a1f166d409fabe6124f71882960e8a67995",
      "12": "5c8180eddac91c0106f98b1ac364c942a306eb89e71dd9cefd5c71c2479f3a5e"
    },
    "nsv/20_nsv_bsiso_refine_analysis.ipynb": {
      "4": "534c9b2503dcd711aa2db34b5bb4ccbbb9e9f265985d68e86b7d65b8d8550fd2",
      "6": "30f74b9a60ac85a4a150b8f5f9765e478dcc4de26b3dc6a3984ef9bb97c431f0",
      "16": "55c385f8bb8c2b6dcfcab2b6a475b8c27b3777d1bb2d6d4363907677dd1536af"
    },
    "mjo/15_mjo_ssl_temporal_2d.ipynb": {
      "2": "6e07d3c80e521e5edc713f2a335c3554cf5b9c3daba2ee5cc88f3c3d3c6752a6",
      "12": "1a5a9d10375fd8ec93881593850977aec4c59c22b0966324775adfc6521a1fa8"
    },
    "mjo/26_mjo_barlow_twins.ipynb": {
      "2": "895ee030c464e27349c67c7206afd169178dbbcb436b844aed35b0e0d161b127",
      "6": "eb084f42a847b96c1ba9b9250466e60d92fdd4ad75681cdf6d87a546f5dd077f",
      "8": "735e209a26864212cf10fd1e9d465a66f77f802c0dcc7066f74367a9ef630a42",
      "10": "7e77178ed1bcd46452a08970be24d260bebbe2d7acd0a5587356dc9bb3b4f27c",
      "22": "55901df7ccfe6bf718879a787f5a65ba50da600976455df16365ca452a9c75ac"
    }
  }
}


## Cell 3 — Condition and training configuration


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from dataclasses import replace
from l1_data import CONDITIONS, load_dataset, frame_cache, observation
from l1_climate import PROTOCOL, ClimateConfig, ClimateModel, climate_config, run_training

DATASET_ID = ''  # paste from notebook 39; blank works only if exactly one dataset exists
CONDITIONS_TO_RUN = ['L1a']  # later: L1b_s002, L1b_s005, L1c_visible, L1c_hidden
SEEDS = [42]  # pilot first; then [42, 43, 44] with the same list in notebook 41
# Each method uses its own reference defaults; do not share one BASE optimizer.
RUN_TAG = 'climate_replica'
DATASET_ID = resolve_dataset_id(DATASET_ID)
if not torch.cuda.is_available():
    raise RuntimeError('Select Colab Runtime > Change runtime type > T4 GPU for training.')
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)
print('Dataset:', DATASET_ID, '| conditions:', CONDITIONS_TO_RUN)
for condition in CONDITIONS_TO_RUN:
    if condition not in CONDITIONS:
        raise ValueError(condition)


## Cell 4 — Predictive encoder, train+validation ID, and NSV refinement


In [ ]:
RUNS = []
for condition in CONDITIONS_TO_RUN:
    for seed in SEEDS:
        config = climate_config('nsv', seed=seed, run_tag=RUN_TAG)
        # Defaults: 100 prediction epochs, 500 refinement, 300 dynamics.
        # cfg.id_backend='skdim' reproduces the pinned climate estimator calls.
        path = run_training(L1_ROOT, DATASET_ID, condition, CACHE_DIR, config)
        RUNS.append(path)
        print('Completed:', path)


## Cell 5 — Validation curves and ID scale sensitivity


In [ ]:
for path in RUNS:
    history = pd.read_json(path / 'training_history.json')
    history.plot(x='epoch', y=['train', 'val'], title=path.name)
    plt.ylabel('Future-observation MSE'); plt.show()
    diagnostic = json.loads((path / 'id_train.json').read_text())
    display(pd.DataFrame(diagnostic['scales']))
    rows = []
    print('Climate ID:', json.loads((path / 'intrinsic_dim.json').read_text()))
    for p in sorted(path.glob('refine_d*_history.json')):
        history = pd.read_json(p)
        rows.append({'dimension': int(p.stem.split('_d')[-1].split('_')[0]), 'best_val_mse': history.val.min()})
    display(pd.DataFrame(rows).sort_values('dimension'))
    if (path / 'dynamics_scores.json').exists():
        print('Latent dynamics:', json.loads((path / 'dynamics_scores.json').read_text()))


## Cell 6 — Inspect future-frame prediction on validation only


In [ ]:
path = RUNS[0]
meta = json.loads((path / 'run.json').read_text())
config = ClimateConfig(**meta['training'])
condition = meta['condition']
bundle, sim, _ = load_dataset(L1_ROOT, DATASET_ID, CONDITIONS[condition]['kind'])
frames = frame_cache(bundle, sim, condition, CACHE_DIR, DATASET_ID)
trajectory = int(np.flatnonzero(bundle['splits'] == 1)[0])
end = config.window + 10
x = observation(frames, trajectory, end, config.window, CONDITIONS[condition]['noise_std'], sim.seed)
y = observation(frames, trajectory, end+config.lag, config.window,
                CONDITIONS[condition]['noise_std'], sim.seed, draw=0)
model = ClimateModel(config).eval()
model.load_state_dict(torch.load(path / 'model_selected.pt', map_location='cpu', weights_only=True))
with torch.no_grad():
    predicted = model(torch.from_numpy(x[None]))[1][0].numpy()
fig, axes = plt.subplots(3, config.window, figsize=(3*config.window, 8), squeeze=False)
for row, (clip, label) in enumerate([(x, 'input'), (y, 'future observation'), (predicted, 'prediction')]):
    for col in range(config.window):
        axes[row, col].imshow(clip[col], cmap='gray', vmin=0, vmax=1)
        axes[row, col].set_title(label); axes[row, col].axis('off')
fig.tight_layout(); fig.savefig(path / 'validation_prediction.png', dpi=130); plt.show()
print('Next: 41 on the same dataset/conditions/history, then 42 for held-out scores.')
